In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favorite_colour: str

/home/robbie/lca-lc-foundations/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favorite_colour(favorite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favorite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favorite_colour": favorite_colour, 
        "messages": [ToolMessage("Successfully updated favorite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "ollama:qwen2.5:7b",
    tools=[update_favorite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favorite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favorite_colour': 'green',
 'messages': [HumanMessage(content='My favorite colour is green', additional_kwargs={}, response_metadata={}, id='5189e3a0-289d-4589-9255-8b37f0a60202'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-06-09T03:22:27.987442911Z', 'done': True, 'done_reason': 'stop', 'total_duration': 605177568, 'load_duration': 53949557, 'prompt_eval_count': 162, 'prompt_eval_duration': 223098665, 'eval_count': 22, 'eval_duration': 294790657, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019eaa67-26b3-7bf2-888d-3540110db1c0-0', tool_calls=[{'name': 'update_favorite_colour', 'args': {'favorite_colour': 'green'}, 'id': '56f14ea4-44f7-4c52-976d-af944047868e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 22, 'total_tokens': 184}),
              ToolMessage(content='Successfully updated favorite colour', name='upd

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favorite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favorite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='7ab80205-12e6-4efa-92de-5cbf7ef87edb'),
              AIMessage(content="Hello! I'm just a digital assistant, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-06-09T03:23:04.865292517Z', 'done': True, 'done_reason': 'stop', 'total_duration': 725329577, 'load_duration': 46273834, 'prompt_eval_count': 163, 'prompt_eval_duration': 174020655, 'eval_count': 34, 'eval_duration': 460529348, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019eaa67-b64b-7491-96f3-8b000146ff87-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 34, 'total_tokens': 197})]}


## Read state

In [ ]:
@tool
def read_favorite_colour(runtime: ToolRuntime) -> str:
    """Read the favorite colour of the user from the state."""
    try:
        return runtime.state["favorite_colour"]
    except KeyError:
        return "No favorite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favorite_colour, read_favorite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)